# CIP-Clay Binding Free Energy: Kd Analysis
## ClayKd · ClayKdImproved — Full Analysis Pipeline

All methods implemented in `ClayPath.py` (`ClayKd` + `ClayKdImproved`):

| Section | Method | What it does |
|---------|--------|--------------|
| 3 | `compute()` | Standard Kd via partition-function integral |
| 3 | `compute(use_regression=True)` | Same with regression-fit bulk reference |
| 4 | `bootstrap_uncertainty()` | Intra-path bootstrap CI on Kd |
| 4 | `report_kd_with_confidence()` | Geometric mean + empirical percentile CI |
| 5 | `compute_surface()` | Surface Kd with explicit binding-site area |
| 5 | `sensitivity_to_area()` | Kd vs A_site sensitivity table |
| 6 | `_check_pmf_convergence()` | Bulk slope / RMS convergence check |
| 7 | `correct_long_range_tail()` | Debye-Hückel tail correction |
| 7 | `compute_entropic_correction()` | Orientational entropy TΔS |
| 7 | `compute_with_entropy_correction()` | Corrected Kd |
| 8 | `compute_orientation_averaged_kd()` | 2D r×θ integral (requires W(r,θ)) |
| 9 | `compute_cation_binding_contribution()` | Competitive cation inhibition |
| 10 | `compute_ensemble_kd()` | Inter-replica geometric mean + CI |

**Usage**: set `CLAY_PATH_NPZ` in **Section 1**, then run all cells in order.

---
*Author: R.Swai — 2026*

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, '/Users/roev0007/Documents/solvation_shells/my_scripts')

from ClayPath import ClayPath, ClayKd, ClayKdImproved

%matplotlib inline
plt.rcParams['figure.dpi']     = 100
plt.rcParams['savefig.dpi']    = 300
plt.rcParams['font.size']      = 11
plt.rcParams['axes.titlesize'] = 12

print('ClayKd          :', ClayKd)
print('ClayKdImproved  :', ClayKdImproved)

## 1. System Setup

Point `CLAY_PATH_NPZ` at the `.npz` file saved by `ClayPath.save()` in the
`ClayFreeEnergy_ExtraAnalysis` notebook (Section 7).  
All parameters below must match the simulation that produced that file.

In [ ]:
# --------------------------------------------------------------------------
#  EDIT THESE
# --------------------------------------------------------------------------

# Path to the ClayPath .npz produced by Section 7 of ExtraAnalysis notebook
CLAY_PATH_NPZ = (
    '/Volumes/My_bckp/project_1_US/US/CIP/with_salts/Negative_with_salt'
    '/CIP_MgCl2_CaCl2/MgCl2/ExtraAnalysis_results/clay_path.npz'
)

# Optional: path to W(r,theta) saved by ClayPMF2D.save_results()
# Required ONLY for Section 8 (orientation-averaged Kd).
# Set to None to skip Section 8.
PMF2D_NPZ = (
    '/Volumes/My_bckp/project_1_US/US/CIP/with_salts/Negative_with_salt'
    '/CIP_MgCl2_CaCl2/MgCl2/ExtraAnalysis_results/pmf2d_W.npz'
)

TEMPERATURE     = 298.15   # K
ION_CONC_M      = 0.1      # mol/L — MgCl2 concentration
ION_CHARGE      = +2       # Mg2+
ION_NAME        = 'Mg'
DEBYE_LENGTH_NM = 0.96     # nm  (0.1 M MgCl2, T=298K)

# Binding-site area for surface Kd (nm^2 per site)
# Typical MMT ditrigonal cavity: ~0.25 nm^2
BINDING_SITE_AREA_NM2 = 0.25

# Experimental Kd for comparison (M); set to None to skip
KD_EXP_M = None   # e.g. 5e-6

# -- Additional PMF sources for method comparison (Section 13) ---------------
# Set any path to None to skip that source.

# MBAR .npz saved by ClayMBAR.save() (Section 4 of ExtraAnalysis notebook)
MBAR_NPZ = (
    '/Volumes/My_bckp/project_1_US/US/CIP/with_salts/Negative_with_salt'
    '/CIP_MgCl2_CaCl2/MgCl2/ExtraAnalysis_results/clay_mbar.npz'
)

# WHAM |z| profile saved by ClayPMF.save_results() -> <prefix>_abs.dat
WHAM_ABS_DAT = (
    '/Volumes/My_bckp/project_1_US/US/CIP/with_salts/Negative_with_salt'
    '/CIP_MgCl2_CaCl2/MgCl2/PMF_results/pmf_python_abs.dat'
)

# gmx wham symmetrised profile (.xvg from 'gmx wham -sym')
GMX_PROFILE = (
    '/Volumes/My_bckp/project_1_US/US/CIP/with_salts/Negative_with_salt'
    '/CIP_MgCl2_CaCl2/MgCl2/sym_profile.xvg'
)

# gmx wham bootstrap profile (set to None to skip)
GMX_BS_PROF = (
    '/Volumes/My_bckp/project_1_US/US/CIP/with_salts/Negative_with_salt'
    '/CIP_MgCl2_CaCl2/MgCl2/sym_bsprofile.xvg'
)

# Energy unit reported by gmx wham ('kT', 'kJ/mol', or 'kcal/mol')
GMX_UNIT = 'kT'

OUTPUT_DIR = os.path.join(
    os.path.dirname(CLAY_PATH_NPZ), 'Kd_Analysis'
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'ClayPath npz : {CLAY_PATH_NPZ}')
print(f'PMF2D npz    : {PMF2D_NPZ}')
print(f'T            : {TEMPERATURE} K')
print(f'Ion          : {ION_NAME}{ION_CHARGE:+d}  {ION_CONC_M} M')
print(f'Debye length : {DEBYE_LENGTH_NM} nm')
print(f'A_site       : {BINDING_SITE_AREA_NM2} nm²')
print(f'Output dir   : {OUTPUT_DIR}')print(f'MBAR npz     : {MBAR_NPZ}')
print(f'WHAM dat     : {WHAM_ABS_DAT}')
print(f'gmx profile  : {GMX_PROFILE}')


## 2. Load ClayPath

In [ ]:
clay_path = ClayPath.load(CLAY_PATH_NPZ)

r    = clay_path.r_path
pmf  = clay_path.pmf_path
n_im = len(r)

print(f'Path images  : {n_im}')
print(f'r range      : [{r.min():.3f}, {r.max():.3f}] nm')
print(f'PMF range    : [{pmf.min():.2f}, {pmf.max():.2f}] kJ/mol')
print(f'Convention   : r[0]={r[0]:.3f} (bulk), r[-1]={r[-1]:.3f} (surface)')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(r, pmf, 'o-', ms=4, lw=1.5, color='steelblue')
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('r  (nm)')
ax.set_ylabel('W(r)  (kJ/mol)')
ax.set_title('PMF along MFEP  (raw, before zero-referencing)')
fig.tight_layout()
plt.show()

## 3. ClayKd — Standard Kd Computation

Two zero-reference strategies are compared:

| Method | Zero reference |
|--------|----------------|
| `use_regression=False` (default) | Mean of first `n_bulk_points` images |
| `use_regression=True` | OLS fit to bulk plateau; evaluated at outermost point |

In [ ]:
# --- 3a. Standard mean-reference Kd ----------------------------------------
kd = ClayKd(clay_path, T=TEMPERATURE)

res_mean = kd.compute(
    n_bulk_points  = 5,
    use_regression = False,
    verbose        = True,
)

In [ ]:
# --- 3b. Regression-reference Kd -------------------------------------------
kd_r = ClayKd(clay_path, T=TEMPERATURE)

res_reg = kd_r.compute(
    n_bulk_points  = 10,   # more points → better fit
    use_regression = True,
    verbose        = True,
)

In [ ]:
# --- 3c. Regression diagnostics --------------------------------------------
if hasattr(kd_r, '_bulk_regression'):
    reg = kd_r._bulk_regression
    print('Bulk regression diagnostics:')
    print(f"  slope      = {reg['slope_kJ_per_nm']:+.4f} kJ/mol/nm")
    print(f"  intercept  = {reg['intercept_kJ']:+.3f} kJ/mol")
    print(f"  R²         = {reg['r_squared']:.4f}")
    print(f"  RMSE       = {reg['rmse_kJ']:.4f} kJ/mol")
    delta = reg['w_ref_kJ'] - reg['w_mean_kJ']
    print(f"  Δ(reg−mean) = {delta:+.4f} kJ/mol  "
          f"(ΔKd ~ {abs(delta)/2.479*100:.1f}% at 298 K)")

In [ ]:
# --- 3d. Side-by-side comparison -------------------------------------------
print(f"{'Method':<22} {'Kd':>14}  {'dG (kJ/mol)':>12}")
print('-' * 52)
for label, res in [('Mean reference', res_mean), ('Regression ref', res_reg)]:
    kd_str = ClayKd._fmt_kd(res['Kd_pf_M'])
    print(f"  {label:<20} {kd_str:>14}  {res['dG_pf_kJ']:>+12.2f}")

In [ ]:
kd.summary()

In [ ]:
# Plot zero-referenced PMF (working copy)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(kd.r, kd.pmf_ref, 'o-', ms=4, lw=1.5, label='Mean ref')
ax.plot(kd_r.r, kd_r.pmf_ref, 's--', ms=4, lw=1.5, label='Regression ref')
ax.axhline(0, color='k', lw=0.8, ls=':')
ax.axvline(kd._results['r_saddle_nm'], color='grey', lw=1, ls='--',
           label=f"saddle r={kd._results['r_saddle_nm']:.3f} nm")
ax.set_xlabel('r  (nm)')
ax.set_ylabel('W(r)  (kJ/mol)')
ax.set_title('Zero-referenced PMF: Mean vs Regression bulk reference')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'pmf_bulk_reference_comparison.png'),
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- 3e. Compare with experiment (if KD_EXP_M set) -------------------------
if KD_EXP_M is not None:
    kd.compare_with_experiment(Kd_exp_M=KD_EXP_M)
else:
    print('KD_EXP_M not set — skipping experimental comparison.')
    print('Set KD_EXP_M in Section 1 and re-run this cell.')

## 4. Bootstrap Uncertainty & Confidence Intervals

`bootstrap_uncertainty` resamples the PMF images with replacement.
`report_kd_with_confidence` reports geometric mean + empirical percentile CI
(no parametric assumption on the Kd distribution).

In [ ]:
# --- 4a. Run bootstrap (cached after first call) ---------------------------
bs = kd.bootstrap_uncertainty(
    n_bootstrap   = 500,
    n_bulk_points = 5,
    verbose       = True,
)

In [ ]:
# --- 4b. 95% and 90% CI report ---------------------------------------------
for cl in (0.95, 0.90):
    kd.report_kd_with_confidence(confidence_level=cl, verbose=True)
    print()

In [ ]:
# --- 4c. Bootstrap histogram -----------------------------------------------
kd_vals = np.array(bs['Kd_pf_M_values'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(kd_vals * 1e3, bins=40, color='steelblue', edgecolor='white', lw=0.5)
axes[0].axvline(res_mean['Kd_pf_M'] * 1e3, color='red', lw=1.5, label='Observed')
axes[0].set_xlabel('Kd  (mM)')
axes[0].set_ylabel('Count')
axes[0].set_title('Bootstrap distribution of Kd')
axes[0].legend()

dg_vals = np.array(bs['dG_pf_kJ_values'])
axes[1].hist(dg_vals, bins=40, color='darkorange', edgecolor='white', lw=0.5)
axes[1].axvline(res_mean['dG_pf_kJ'], color='red', lw=1.5, label='Observed')
axes[1].set_xlabel('ΔG°  (kJ/mol)')
axes[1].set_ylabel('Count')
axes[1].set_title('Bootstrap distribution of ΔG°')
axes[1].legend()

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'bootstrap_distributions.png'),
            dpi=300, bbox_inches='tight')
plt.show()

## 5. Surface Binding Kd & Area Sensitivity

`compute_surface` uses the explicit binding-site area $A_{\rm site}$:  
$$K_d^{\rm surf} = \frac{1}{c^\circ \cdot A_{\rm site} \cdot I_{\rm bound}}$$

In [ ]:
# --- 5a. Surface Kd at chosen A_site ---------------------------------------
surf = kd.compute_surface(
    binding_site_area_nm2 = BINDING_SITE_AREA_NM2,
    verbose               = True,
)

In [ ]:
# --- 5b. Sensitivity: how much does Kd change with A_site? ----------------
kd.sensitivity_to_area(
    A_min_nm2 = 0.10,
    A_max_nm2 = 1.00,
    n_points  = 12,
    verbose   = True,
)

In [ ]:
# Plot Kd_surf vs A_site
A_arr  = np.logspace(np.log10(0.05), np.log10(2.0), 80)
kd_arr = np.array([
    1.0 / (kd._C_STD * a * kd._results['I_bound_nm'])
    for a in A_arr
])

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(A_arr, kd_arr * 1e3, 'b-', lw=2)
ax.axvline(BINDING_SITE_AREA_NM2, color='red', ls='--',
           label=f'A_site = {BINDING_SITE_AREA_NM2} nm²')
ax.set_xlabel('Binding site area  (nm²)')
ax.set_ylabel('K_d^surf  (mM)')
ax.set_title('Surface Kd vs A_site')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'surface_kd_vs_area.png'),
            dpi=300, bbox_inches='tight')
plt.show()

## 6. PMF Convergence Check

`_check_pmf_convergence` examines the bulk plateau:  
- linear slope (should be ≈ 0)  
- RMS fluctuations (should be < kT ≈ 2.5 kJ/mol)

In [ ]:
conv_check = kd._check_pmf_convergence(
    n_bulk_points = 10,
    verbose       = True,
)

## 7. ClayKdImproved — Tail Correction + Entropy

`ClayKdImproved` adds four physics-based corrections on top of `ClayKd`.

**Tail correction**: a Debye-Hückel taper fills the missing PMF beyond the
last sampled image out to $r = 5\lambda_D$, reducing systematic underestimation
of $I_{\rm bound}$ when sampling does not fully reach bulk.  

**Entropy correction**: accounts for lost translational degrees of freedom
when CIP adsorbs from bulk solution.

In [ ]:
# Create ClayKdImproved and run standard compute first
ki = ClayKdImproved(clay_path, T=TEMPERATURE)
ki.compute(n_bulk_points=5, verbose=False)

In [ ]:
# --- 7a. Tail correction ---------------------------------------------------
tail = ki.correct_long_range_tail(
    ion_charge       = ION_CHARGE,
    ion_conc_M       = ION_CONC_M,
    debye_length_nm  = DEBYE_LENGTH_NM,
    r_extension_nm   = 5.0 * DEBYE_LENGTH_NM,
    verbose          = True,
)

In [ ]:
# Before/after tail correction comparison
print(f"I_bound before correction : {tail['I_bound_before_nm']:.6f} nm")
print(f"I_bound after  correction : {tail['I_bound_after_nm']:.6f} nm")
print(f"Kd before                 : {ClayKd._fmt_kd(tail['Kd_before_M'])}")
print(f"Kd after                  : {ClayKd._fmt_kd(tail['Kd_after_M'])}")

# Plot PMF + tail extension
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ki.r, ki.pmf_ref, 'o-', ms=3.5, lw=1.5, label='Sampled PMF')
if 'r_tail' in tail and 'w_tail' in tail:
    ax.plot(tail['r_tail'], tail['w_tail'], '--', lw=1.5, color='red',
            label='DH tail extension')
ax.axhline(0, color='k', lw=0.8, ls=':')
ax.set_xlabel('r  (nm)')
ax.set_ylabel('W(r)  (kJ/mol)')
ax.set_title('PMF with Debye-Hückel tail correction')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'tail_correction.png'),
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- 7b. Entropic correction -----------------------------------------------
ent = ki.compute_entropic_correction(verbose=True)

In [ ]:
# --- 7c. Kd with entropy correction ----------------------------------------
ent_res = ki.compute_with_entropy_correction(verbose=True)

## 8. Orientation-Averaged Kd from 2D W(r, θ)

Integrates the full 2D surface with the sin(θ) Jacobian:

$$I_{\rm bound}^{\rm 2D} = \int_{r_{\rm min}}^{r_{\rm saddle}}
  \int_0^{\pi/2} e^{-W(r,\theta)/RT} \sin\theta\, d\theta\, dr$$

Requires `ClayPMF2D` with W(r,θ) already computed.

In [ ]:
if PMF2D_NPZ and os.path.exists(PMF2D_NPZ):
    from ClayPMF2D import ClayPMF2D
    pmf2d = ClayPMF2D.load_results(PMF2D_NPZ)
    print(f'W(r,\u03b8) loaded: shape = {pmf2d.pmf_2d.shape}')

    ki2d = ClayKdImproved(clay_path, T=TEMPERATURE)
    ki2d.compute(n_bulk_points=5, verbose=False)

    res_2d = ki2d.compute_orientation_averaged_kd(
        pmf2d    = pmf2d,
        r_max    = ki2d._results['r_saddle_nm'],
        verbose  = True,
    )
else:
    print('PMF2D_NPZ not found — skipping Section 8.')
    print(f'Expected: {PMF2D_NPZ}')

## 9. Competitive Cation Binding

Uses a competitive Langmuir + ΔΔG model to estimate how cation pre-binding
to the clay surface modulates CIP adsorption.

Default binding constants:

| Ion | K_clay (M⁻¹) | ΔΔG (kJ/mol) |
|-----|------------|-------------|
| Na  | 1.0        | −3.0        |
| K   | 3.0        | −4.0        |
| Mg  | 30.0       | −10.0       |
| Ca  | 50.0       | −8.0        |

In [ ]:
ki_cat = ClayKdImproved(clay_path, T=TEMPERATURE)
ki_cat.compute(n_bulk_points=5, verbose=False)

cat_res = ki_cat.compute_cation_binding_contribution(
    ion         = ION_NAME,
    ion_conc_M  = ION_CONC_M,
    verbose     = True,
)

In [ ]:
# Plot Kd_eff vs cation concentration
conc_arr  = np.logspace(-3, 1, 100)   # 1 mM to 10 M
K_clay    = ClayKdImproved._CLAY_K_DEFAULT.get(ION_NAME, 1.0)
ddG       = ClayKdImproved._CLAY_DDG_DEFAULT.get(ION_NAME, 0.0)
RT        = 8.314462618e-3 * TEMPERATURE
theta_arr = K_clay * conc_arr / (1.0 + K_clay * conc_arr)   # site occupancy
Kd0       = ki_cat._results['Kd_pf_M']
Kd_eff    = Kd0 * np.exp(-theta_arr * ddG / RT)

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(conc_arr, Kd_eff * 1e3, 'b-', lw=2)
ax.axvline(ION_CONC_M, color='red', ls='--',
           label=f'[{ION_NAME}] = {ION_CONC_M} M')
ax.axhline(Kd0 * 1e3, color='grey', ls=':', lw=1, label='No cation')
ax.set_xlabel(f'[{ION_NAME}²⁺]  (M)')
ax.set_ylabel('Effective K_d  (mM)')
ax.set_title(f'Competitive {ION_NAME}²⁺ effect on CIP-clay K_d')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'cation_competitive_kd.png'),
            dpi=300, bbox_inches='tight')
plt.show()

## 10. Ensemble Kd from Multiple Independent Paths

`compute_ensemble_kd` captures **inter-replica variability** — how much Kd
changes between independent MFEP runs — which is distinct from the
intra-path bootstrap variability in Section 4.

- Uses Student-t CI on log(Kd) for n < 30 replicas
- Returns geometric mean + geometric std factor

In [ ]:
# --------------------------------------------------------------------------
#  Set REPLICA_NPZS to a list of clay_path.npz files from independent runs.
#  Leave as empty list to skip.
# --------------------------------------------------------------------------
REPLICA_NPZS = [
    # '/path/to/replica1/clay_path.npz',
    # '/path/to/replica2/clay_path.npz',
    # '/path/to/replica3/clay_path.npz',
]

if len(REPLICA_NPZS) >= 2:
    replica_paths = [ClayPath.load(p) for p in REPLICA_NPZS]
    ki_list = [ClayKdImproved(cp, T=TEMPERATURE) for cp in replica_paths]

    ens = ClayKdImproved.compute_ensemble_kd(
        ki_list,
        n_bulk_points    = 5,
        use_regression   = False,
        confidence_level = 0.95,
        verbose          = True,
    )
else:
    print('REPLICA_NPZS not set — skipping ensemble Kd.')
    print('Add paths to REPLICA_NPZS (at least 2 replicas) and re-run.')

In [ ]:
# Ensemble visualisation (runs only when replicas are loaded)
if len(REPLICA_NPZS) >= 2:
    kd_vals_ens = np.array(ens['Kd_values_M']) * 1e3   # mM

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(range(1, len(kd_vals_ens) + 1), kd_vals_ens,
           color='steelblue', edgecolor='white', width=0.6)
    ax.axhline(ens['Kd_geom_mean_M'] * 1e3,
               color='red', lw=2, label='Geometric mean')
    ax.axhspan(ens['Kd_ci_lower_M'] * 1e3,
               ens['Kd_ci_upper_M'] * 1e3,
               alpha=0.15, color='red', label='95% CI')
    ax.set_xlabel('Replica')
    ax.set_ylabel('Kd  (mM)')
    ax.set_title(f'Ensemble Kd  ({len(kd_vals_ens)} replicas)')
    ax.set_xticks(range(1, len(kd_vals_ens) + 1))
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, 'ensemble_kd.png'),
                dpi=300, bbox_inches='tight')
    plt.show()

## 11. PMF Method Comparison — MBAR · WHAM · gmx

Compare the Kd estimate from the string-method MFEP (Sections 3–7) against three
independent 1-D PMF profiles obtained by different free-energy methods.

**Coordinate convention note.** `ClayPMF`, `ClayMBAR`, and `gmx wham` store their
profiles in **|z|-from-box-centre** coordinates:

| r value | physical location |
|---------|-------------------|
| r ≈ 0 | box centre = **bulk** |
| r = r_max | clay surface = **binding site** |

`ClayKd` requires **distance-from-clay-surface** coordinates (r = 0 at clay,
large r at bulk). The `_PMFStub` adapter applies the linear transform
`r_clay = r_max − r` so that `r_clay[0]` = r_max (bulk, W ≈ 0) and
`r_clay[-1]` = 0 (surface, W < 0).

In [ ]:
# -- _PMFStub: adapter from |z|-from-center to ClayKd convention ------------

class _PMFStub:
    """
    Minimal duck-type adapter so ClayKd can consume any 1-D (r, W) profile.

    ClayPMF, ClayMBAR, and gmx wham store the PMF in |z|-from-centre
    coordinates: r=0 -> bulk (box centre), r_max -> clay surface.
    ClayKd requires distance-from-clay coords: index 0 = bulk (large r).

    Transform applied:  r_clay = r_max - r_pmf
      r_clay[0] = r_max  (bulk,    W[0] ~= 0)   <- ClayKd index 0
      r_clay[-1] = 0     (surface, W[-1] < 0)   <- ClayKd index -1
    """
    def __init__(self, r_pmf, w_pmf):
        r = np.asarray(r_pmf, dtype=float)
        w = np.asarray(w_pmf, dtype=float)
        ok = np.isfinite(r) & np.isfinite(w)
        r, w = r[ok], w[ok]
        r_max = float(r.max())
        self.r_path   = r_max - r   # r_clay: large at bulk, 0 at surface
        self.pmf_path = w.copy()    # W values unchanged


def _kd_from_profile(r_pmf, w_pmf, T, verbose=True):
    """Build _PMFStub -> ClayKd -> compute(); return (kd_obj, result_dict)."""
    stub = _PMFStub(r_pmf, w_pmf)
    kd_obj = ClayKd(stub, T=T)
    res = kd_obj.compute(n_bulk_points=5, verbose=verbose)
    return kd_obj, res


# Storage for raw profile arrays (used by the comparison plot cell)
_r_mbar_raw = _w_mbar_raw = None
_r_wham_raw = _w_wham_raw = None
_r_gmx_raw  = _w_gmx_raw  = None

print('_PMFStub adapter and _kd_from_profile helper defined.')

In [ ]:
# -- 13a. MBAR Kd ------------------------------------------------------------
res_mbar = None
kd_mbar  = None

_mbar_path = MBAR_NPZ if isinstance(MBAR_NPZ, str) else None
if _mbar_path and os.path.isfile(_mbar_path):
    from ClayMBAR import ClayMBAR as _ClayMBAR
    _mbar_obj    = _ClayMBAR.load(_mbar_path)
    _r_mbar_raw  = _mbar_obj.bin_centers_abs   # nm, bulk at r~0
    _w_mbar_raw  = _mbar_obj.pmf_mbar_1d       # kJ/mol
    print(f'MBAR profile loaded: {len(_r_mbar_raw)} bins, '
          f'r=[{_r_mbar_raw.min():.3f}, {_r_mbar_raw.max():.3f}] nm')
    kd_mbar, res_mbar = _kd_from_profile(_r_mbar_raw, _w_mbar_raw,
                                          TEMPERATURE, verbose=True)
    print(f'\nMBAR  Kd = {ClayKd._fmt_kd(res_mbar["Kd_pf_M"])}')
    print(f'MBAR  dG = {res_mbar["dG_pf_kJ"]:+.2f} kJ/mol')
else:
    print(f'MBAR_NPZ not found or not set -- skipping.')
    print(f'  path: {_mbar_path}')

In [ ]:
# -- 13b. WHAM 1-D Kd --------------------------------------------------------
res_wham = None
kd_wham  = None

_wham_path = WHAM_ABS_DAT if isinstance(WHAM_ABS_DAT, str) else None
if _wham_path and os.path.isfile(_wham_path):
    _wham_data   = np.loadtxt(_wham_path, comments='#')
    _r_wham_raw  = _wham_data[:, 0]   # nm, bulk at r~0
    _w_wham_raw  = _wham_data[:, 1]   # kJ/mol
    print(f'WHAM profile loaded: {len(_r_wham_raw)} bins, '
          f'r=[{_r_wham_raw.min():.3f}, {_r_wham_raw.max():.3f}] nm')
    kd_wham, res_wham = _kd_from_profile(_r_wham_raw, _w_wham_raw,
                                          TEMPERATURE, verbose=True)
    print(f'\nWHAM  Kd = {ClayKd._fmt_kd(res_wham["Kd_pf_M"])}')
    print(f'WHAM  dG = {res_wham["dG_pf_kJ"]:+.2f} kJ/mol')
else:
    print(f'WHAM_ABS_DAT not found or not set -- skipping.')
    print(f'  path: {_wham_path}')

In [ ]:
# -- 13c. gmx wham Kd --------------------------------------------------------
res_gmx = None
kd_gmx  = None

_RT         = 8.314462618e-3 * TEMPERATURE        # kJ/mol at TEMPERATURE
_unit_map   = {'kT': _RT, 'kJ/mol': 1.0, 'kcal/mol': 4.184}
_gmx_scale  = _unit_map.get(GMX_UNIT, 1.0)

_gmx_path = GMX_PROFILE if isinstance(GMX_PROFILE, str) else None
if _gmx_path and os.path.isfile(_gmx_path):
    _gmx_data   = np.loadtxt(_gmx_path, comments=['#', '@'])
    _r_gmx_raw  = _gmx_data[:, 0]                    # nm
    _w_gmx_raw  = _gmx_data[:, 1] * _gmx_scale       # -> kJ/mol
    print(f'gmx profile loaded: {len(_r_gmx_raw)} bins, '
          f'r=[{_r_gmx_raw.min():.3f}, {_r_gmx_raw.max():.3f}] nm')
    kd_gmx, res_gmx = _kd_from_profile(_r_gmx_raw, _w_gmx_raw,
                                        TEMPERATURE, verbose=True)
    print(f'\ngmx   Kd = {ClayKd._fmt_kd(res_gmx["Kd_pf_M"])}')
    print(f'gmx   dG = {res_gmx["dG_pf_kJ"]:+.2f} kJ/mol')
else:
    print(f'GMX_PROFILE not found or not set -- skipping.')
    print(f'  path: {_gmx_path}')

In [ ]:
# -- 13d. PMF overlay: all profiles in distance-from-clay coordinates --------
# After r_clay = r_max - r all profiles share the same x-axis:
# 0 = clay surface,  large r = bulk.

fig, ax = plt.subplots(figsize=(9, 4.5))

# 1. MFEP profile (already in dist-from-clay coords: r[0]=bulk=large)
_r_mfep = clay_path.r_path
_w_mfep = kd.w_zeroed if hasattr(kd, 'w_zeroed') else clay_path.pmf_path
ax.plot(_r_mfep, _w_mfep, 'o-', ms=4, lw=2.0, color='steelblue',
        label='MFEP (string method)', zorder=4)

# 2. MBAR
if _r_mbar_raw is not None:
    _ok = np.isfinite(_r_mbar_raw) & np.isfinite(_w_mbar_raw)
    ax.plot(_r_mbar_raw.max() - _r_mbar_raw[_ok], _w_mbar_raw[_ok],
            '-', lw=1.8, color='darkorange', label='MBAR', alpha=0.85)

# 3. WHAM
if _r_wham_raw is not None:
    _ok = np.isfinite(_r_wham_raw) & np.isfinite(_w_wham_raw)
    ax.plot(_r_wham_raw.max() - _r_wham_raw[_ok], _w_wham_raw[_ok],
            '--', lw=1.8, color='forestgreen', label='WHAM (Python)', alpha=0.85)

# 4. gmx
if _r_gmx_raw is not None:
    _ok = np.isfinite(_r_gmx_raw) & np.isfinite(_w_gmx_raw)
    ax.plot(_r_gmx_raw.max() - _r_gmx_raw[_ok], _w_gmx_raw[_ok],
            ':', lw=2.0, color='crimson', label='gmx WHAM', alpha=0.85)

ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('Distance from clay surface  (nm)')
ax.set_ylabel('W(r)  (kJ/mol)')
ax.set_title('PMF profiles — method comparison')
ax.legend(fontsize=9)
fig.tight_layout()
_out = os.path.join(OUTPUT_DIR, 'pmf_method_comparison.png')
plt.savefig(_out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {_out}')

In [ ]:
# -- 13e. Kd comparison bar chart across all PMF methods ---------------------
_method_results = [
    ('MFEP (mean ref)',  res_mean),
    ('MFEP (reg. ref)',  res_reg),
    ('MBAR',            res_mbar),
    ('WHAM (Python)',   res_wham),
    ('gmx WHAM',        res_gmx),
]
_avail = [(lab, r) for lab, r in _method_results if r is not None]

if _avail:
    _labels  = [lab for lab, _ in _avail]
    _kd_vals = [r['Kd_pf_M']  for _, r in _avail]
    _dg_vals = [r['dG_pf_kJ'] for _, r in _avail]
    _colors  = ['steelblue', 'cornflowerblue', 'darkorange',
                'forestgreen', 'crimson'][:len(_avail)]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].bar(_labels, _kd_vals, color=_colors, edgecolor='k', linewidth=0.6)
    axes[0].set_yscale('log')
    axes[0].set_ylabel('Kd  (M)')
    axes[0].set_title('Binding affinity by PMF method')
    axes[0].tick_params(axis='x', rotation=20)

    axes[1].bar(_labels, _dg_vals, color=_colors, edgecolor='k', linewidth=0.6)
    axes[1].axhline(0, color='k', lw=0.8)
    axes[1].set_ylabel('ΔG_bind  (kJ/mol)')
    axes[1].set_title('Binding free energy by PMF method')
    axes[1].tick_params(axis='x', rotation=20)

    fig.tight_layout()
    _out = os.path.join(OUTPUT_DIR, 'kd_method_comparison.png')
    plt.savefig(_out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {_out}')
    print()
    print(f'{"Method":<22}  {"Kd":>14}  {"dG (kJ/mol)":>12}')
    print('-' * 54)
    for lab, res in _avail:
        kd_str = ClayKd._fmt_kd(res['Kd_pf_M'])
        print(f'  {lab:<20}  {kd_str:>14}  {res["dG_pf_kJ"]:+.2f}')
else:
    print('No PMF comparison results (MBAR / WHAM / gmx all skipped).')

## 12. Full Summary

Collect all Kd estimates in one table for easy comparison.

In [ ]:
rows = []

rows.append(('ClayKd (mean ref)',          res_mean['Kd_pf_M'], res_mean['dG_pf_kJ']))
rows.append(('ClayKd (regression ref)',    res_reg['Kd_pf_M'],  res_reg['dG_pf_kJ']))

if KD_EXP_M is not None:
    rows.append(('Experimental',           KD_EXP_M,            None))

if 'Kd_after_M' in tail:
    rows.append(('After tail correction',  tail['Kd_after_M'],  None))

if ent_res and 'Kd_corrected_M' in ent_res:
    rows.append(('+ entropy correction',   ent_res['Kd_corrected_M'], ent_res.get('dG_corrected_kJ')))

if len(REPLICA_NPZS) >= 2:
    rows.append(('Ensemble geom. mean',   ens['Kd_geom_mean_M'], ens['dG_mean_kJ']))

# PMF method comparison (Section 13)
if res_mbar is not None:
    rows.append(('MBAR PMF',           res_mbar['Kd_pf_M'], res_mbar['dG_pf_kJ']))
if res_wham is not None:
    rows.append(('WHAM (Python PMF)',  res_wham['Kd_pf_M'], res_wham['dG_pf_kJ']))
if res_gmx is not None:
    rows.append(('gmx WHAM PMF',       res_gmx['Kd_pf_M'],  res_gmx['dG_pf_kJ']))


print(f"{'Method':<35}  {'Kd':>14}  {'dG (kJ/mol)':>12}")
print('-' * 68)
for name, kd_val, dg_val in rows:
    kd_str = ClayKd._fmt_kd(kd_val)
    dg_str = f"{dg_val:+.2f}" if dg_val is not None else '  —   '
    print(f"  {name:<33}  {kd_str:>14}  {dg_str:>12}")

## 13. Save Results

In [ ]:
import json

summary = {
    'Kd_mean_ref_M'       : float(res_mean['Kd_pf_M']),
    'dG_mean_ref_kJ'      : float(res_mean['dG_pf_kJ']),
    'Kd_regression_ref_M' : float(res_reg['Kd_pf_M']),
    'dG_regression_ref_kJ': float(res_reg['dG_pf_kJ']),
    'T_K'                 : TEMPERATURE,
    'ion'                 : ION_NAME,
    'ion_conc_M'          : ION_CONC_M,
    'binding_site_area_nm2': BINDING_SITE_AREA_NM2,
    'n_bootstrap'         : bs['n_bootstrap'],
    'Kd_bs_geom_mean_M'   : float(bs['Kd_geom_M']),
    'Kd_bs_ci_95_lower_M' : None,
    'Kd_bs_ci_95_upper_M' : None,
}

# Attach 95% CI from confidence report
_cr = kd.report_kd_with_confidence(confidence_level=0.95, verbose=False)
summary['Kd_bs_ci_95_lower_M'] = float(_cr['ci_lower_M'])
summary['Kd_bs_ci_95_upper_M'] = float(_cr['ci_upper_M'])

if 'Kd_after_M' in tail:
    summary['Kd_tail_corrected_M'] = float(tail['Kd_after_M'])

if ent_res and 'Kd_corrected_M' in ent_res:
    summary['Kd_entropy_corrected_M'] = float(ent_res['Kd_corrected_M'])

if len(REPLICA_NPZS) >= 2:
    summary['Kd_ensemble_geom_M']  = float(ens['Kd_geom_mean_M'])
    summary['Kd_ensemble_ci_lo_M'] = float(ens['Kd_ci_lower_M'])
    summary['Kd_ensemble_ci_hi_M'] = float(ens['Kd_ci_upper_M'])


# PMF method comparison (Section 13)
if res_mbar is not None:
    summary['Kd_mbar_M']  = float(res_mbar['Kd_pf_M'])
    summary['dG_mbar_kJ'] = float(res_mbar['dG_pf_kJ'])
if res_wham is not None:
    summary['Kd_wham_M']  = float(res_wham['Kd_pf_M'])
    summary['dG_wham_kJ'] = float(res_wham['dG_pf_kJ'])
if res_gmx is not None:
    summary['Kd_gmx_M']   = float(res_gmx['Kd_pf_M'])
    summary['dG_gmx_kJ']  = float(res_gmx['dG_pf_kJ'])

out_json = os.path.join(OUTPUT_DIR, 'kd_summary.json')
with open(out_json, 'w') as fh:
    json.dump(summary, fh, indent=2)

print(f'Summary JSON saved: {out_json}')
print()
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {f:<45}  {size_kb:6.1f} kB')

---
## Reload Saved Objects

Start a fresh session from saved results without re-running the analysis.

In [ ]:
# import json
# summary2 = json.load(open(os.path.join(OUTPUT_DIR, 'kd_summary.json')))
# clay_path2 = ClayPath.load(CLAY_PATH_NPZ)
# kd2 = ClayKd(clay_path2, T=TEMPERATURE)
# kd2.compute(verbose=False)
print('Uncomment lines above to reload saved results.')